In [14]:
import sqlite3
import pandas as pd
import numpy as np
import glob

In [16]:
# import all the file paths
files = glob.glob("../data/raw/*.csv")
files

['../data/raw/customers.csv',
 '../data/raw/accounts.csv',
 '../data/raw/consents.csv',
 '../data/raw/feed_logs.csv',
 '../data/raw/transactions.csv']

In [17]:
conn = sqlite3.connect("cdr_monitor.db")

In [18]:
# save files into sql db

for file in files:
    table_name = 'raw_' + file.split("/")[-1].replace(".csv", "")  
    df = pd.read_csv(file)
    df.to_sql(table_name, conn, if_exists="replace", index=False)


In [19]:
tables_df = pd.read_sql("""
SELECT name 
FROM sqlite_master 
WHERE type='table';
""", conn)

tables_df

,name
0,raw_customers
1,raw_accounts
2,raw_consents
3,raw_feed_logs
4,raw_transactions


In [20]:
# validate if the tables are successfully created in sql db using assert function
table_names= ['raw_customers', 'raw_accounts', 'raw_consents', 'raw_feed_logs', 'raw_transactions']

sql_df = pd.read_sql("""
    SELECT name 
    FROM sqlite_master 
    WHERE type='table';
    """, conn)
sql_table_names = sql_df['name'].to_list()

assert set(table_names) == set(sql_table_names)

In [21]:
# validate if all columns have been inserted into sql tables using assert function

total_row = pd.read_sql("""
SELECT SUM(cnt) AS total_rows
FROM (
    SELECT COUNT(*) AS cnt FROM raw_customers
    UNION ALL
    SELECT COUNT(*) FROM raw_accounts
    UNION ALL
    SELECT COUNT(*) FROM raw_consents
    UNION ALL
    SELECT COUNT(*) FROM raw_feed_logs
    UNION ALL
    SELECT COUNT(*) FROM raw_transactions
)
""", conn)

actual_row = 200+320+280+400+5000

assert total_row.iloc[0,0] == actual_row

In [22]:
conn.close()